# 01. Модуль 1: отделение голоса от фона

Две архитектуры (U-Net на спектрограмме, Conv-TasNet на сигнале) и три функции потерь (L1 по спектру, SI-SDR, multi-resolution STFT). Чекпойнт сохраняется после каждой эпохи; повторный запуск ячейки продолжает обучение.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!test -d /content/call-audio-analysis || git clone -q https://github.com/tanyalebedeva/call-audio-analysis.git /content/call-audio-analysis
%cd /content/call-audio-analysis
!git pull -q
!pip install -q -r requirements.txt

DATA = '/content/drive/MyDrive/coursework/data'
RUNS = '/content/drive/MyDrive/coursework/runs'
REPORTS = '/content/drive/MyDrive/coursework/reports'
!mkdir -p $DATA $RUNS $REPORTS
!nvidia-smi -L

## Обучение

In [ ]:
!python -m callaudio.train --config configs/separation/unet_sisdr.yaml

In [ ]:
!python -m callaudio.train --config configs/separation/unet_l1.yaml

In [ ]:
!python -m callaudio.train --config configs/separation/unet_mrstft.yaml

In [ ]:
!python -m callaudio.train --config configs/separation/tasnet_sisdr.yaml

In [ ]:
!python -m callaudio.train --config configs/separation/tasnet_l1.yaml

In [ ]:
!python -m callaudio.train --config configs/separation/tasnet_mrstft.yaml

## Оценка на тесте

SI-SDRi, PESQ, STOI по уровням SNR и типам фона; WER Whisper на чистой, зашумлённой и очищенной речи.

In [ ]:
import glob
for ckpt in sorted(glob.glob(f'{RUNS}/sep_*/best.pt')):
    !python -m callaudio.evaluate separation --ckpt {ckpt} --limit 300 --wer --whisper small

In [ ]:
!python -m callaudio.report --runs $RUNS --out $REPORTS
from IPython.display import Markdown
Markdown(open(f'{REPORTS}/separation.md', encoding='utf-8').read())

## Кривые обучения

In [ ]:
import json
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
for run in sorted(glob.glob(f'{RUNS}/sep_*')):
    rows = [json.loads(l) for l in open(f'{run}/metrics.jsonl')]
    ax.plot([r['epoch'] for r in rows], [r['val_si_sdri'] for r in rows], marker='o', label=run.split('/')[-1])
ax.set_xlabel('эпоха'); ax.set_ylabel('SI-SDRi на валидации, дБ'); ax.legend(); ax.grid(alpha=.3)
plt.savefig(f'{REPORTS}/separation_curves.png', dpi=150); plt.show()

## Примеры: смесь, очищенная речь, эталон

In [ ]:
from IPython.display import Audio, display
for f in sorted(glob.glob(f'{RUNS}/sep_unet_sisdr/examples/*.wav'))[:6]:
    print(f.split('/')[-1]); display(Audio(f))

## Фильтры энкодера Conv-TasNet

In [ ]:
import torch
from callaudio.tasks import load_trained
model, _, _ = load_trained(f'{RUNS}/sep_tasnet_sisdr/best.pt')
w = model.encoder.weight.detach()[:, 0]
order = torch.fft.rfft(w, n=256).abs().argmax(-1).argsort()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(w[order], aspect='auto', cmap='RdBu'); axes[0].set_title('Фильтры, по доминирующей частоте')
axes[1].imshow(torch.fft.rfft(w[order], n=256).abs(), aspect='auto', origin='lower', cmap='magma'); axes[1].set_title('Амплитудные спектры фильтров')
plt.savefig(f'{REPORTS}/tasnet_filters.png', dpi=150); plt.show()